# Live Multi-Server MCP Demo (VS Code + uv)

One agent, three MCP servers, one question box. This is the short, standalone companion to
`MCP_in_LangChain_CineBot_VSCode.ipynb` — same `uv` environment, same local CineBot server, plus
**two real remote servers**: a free public one (Context7) and your own deployed one
(`time-track-mcp-server`).

```mermaid
graph LR
    U[You type a question] --> AG[Agent]
    AG -->|stdio| S1[(CineBot<br/>local script)]
    AG -->|https, no key| S2[(Context7<br/>mcp.context7.com)]
    AG -->|https| S3[(Your time-track server<br/>vercel.app)]
```


---
## 1. Setup

Uses the same environment as the main notebook. If you haven't already:

```bash
uv sync
cp .env.example .env   # fill in OPENAI_API_KEY
```

Open this notebook in VS Code on the same `.venv` kernel.


In [1]:
import os
import warnings
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), (
    "Missing OPENAI_API_KEY -- copy .env.example to .env and fill in your key."
)

from langchain.mcp import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain.mcp import MCPAdapter
from langchain.agents import create_agent
from fastmcp.client import Client
from fastmcp.client.transports import PythonStdioTransport

print("Environment ready.")


Environment ready.


/var/folders/gd/gpz62qhd2k59xc_wfb3qp1f80000gn/T/ipykernel_38553/1969522913.py:11: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  from langchain.mcp import LangChainBetaWarning


---
## 2. The local server (verified)

Same CineBot server as the main notebook -- written here too so this notebook is self-contained.


In [2]:
%%writefile cinebot_mcp_server.py
from fastmcp import FastMCP

mcp = FastMCP("CineBot")

@mcp.tool()
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie."""
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found.")

if __name__ == "__main__":
    mcp.run()


Overwriting cinebot_mcp_server.py


---
## 3. Connect to each server independently

Rather than one `ClientGroup` (shown in the main notebook), this demo connects to each server in
its **own `try/except`** and merges whatever tool lists succeed. For a live audience, that means
one server being down or network-blocked doesn't take out the whole demo -- you can show "here's
what we got from the two reachable servers" instead of a single crashed cell.


In [3]:
all_tools = []
server_status = {}

# -- 1. Local CineBot (verified by a real run when this notebook was authored) --
try:
    cinebot_transport = PythonStdioTransport(
        script_path=Path("cinebot_mcp_server.py"),
        log_file=Path("cinebot_mcp_server.log"),
    )
    async with MCPAdapter(cinebot_transport) as adapter:
        tools = await adapter.list_tools()
        all_tools.extend(tools)
        server_status["cinebot (local)"] = f"OK -- {[t.name for t in tools]}"
except Exception as e:
    server_status["cinebot (local)"] = f"FAILED -- {type(e).__name__}: {e}"

for name, status in server_status.items():
    print(f"{name}: {status}")


cinebot (local): OK -- ['check_showtimes']


In [ ]:
# -- 2. Context7 (free, public, documented to work keyless at lower rate limits) --
# https://mcp.context7.com/mcp -- pass CONTEXT7_API_KEY as a bearer token if you have one, for
# higher rate limits; works without one for a quick demo.
context7_key = os.environ.get("CONTEXT7_API_KEY")
try:
    context7_client = (
        Client("https://mcp.context7.com/mcp", auth=context7_key)
        if context7_key
        else Client("https://mcp.context7.com/mcp")
    )
    async with MCPAdapter(context7_client) as adapter:
        tools = await adapter.list_tools()
        all_tools.extend(tools)
        server_status["context7 (remote, free)"] = f"OK -- {[t.name for t in tools]}"
except Exception as e:
    server_status["context7 (remote, free)"] = f"FAILED -- {type(e).__name__}: {e}"

print(f"context7 (remote, free): {server_status['context7 (remote, free)']}")


In [4]:
# -- 3. Your own remote server --
# Swap this URL for your own deployment. No API key assumed here -- add one via `auth=` the same
# way as the Context7 cell above if your server needs it.
TIMETRACK_URL = "https://time-track-mcp-server.vercel.app/mcp/"

try:
    async with MCPAdapter(TIMETRACK_URL) as adapter:
        tools = await adapter.list_tools()
        all_tools.extend(tools)
        server_status["time-track (your remote server)"] = f"OK -- {[t.name for t in tools]}"
except Exception as e:
    server_status["time-track (your remote server)"] = f"FAILED -- {type(e).__name__}: {e}"

print(f"time-track (your remote server): {server_status['time-track (your remote server)']}")

print("\n--- Fleet summary ---")
for name, status in server_status.items():
    print(f"{name}: {status}")
print(f"\nTotal tools available to the agent: {len(all_tools)}")


time-track (your remote server): OK -- ['log_time', 'get_timesheet', 'get_project_summary', 'list_projects']

--- Fleet summary ---
cinebot (local): OK -- ['check_showtimes']
time-track (your remote server): OK -- ['log_time', 'get_timesheet', 'get_project_summary', 'list_projects']

Total tools available to the agent: 5


> ⚙️ **Teaching note:** show the fleet summary on screen for a second — it's the whole pitch of
> MCP in one printout: three completely independent services (a local script, a free SaaS tool,
> your own backend), none of which the agent's code knows anything about beyond "call this tool,"
> discovered and merged into one list. Whichever connect, the agent below gets all of them.


---
## 4. Build the agent over whatever connected


In [5]:
assert all_tools, "No server connected -- check the fleet summary above before continuing."

agent = create_agent(model="openai:gpt-5-mini", tools=all_tools)
print(f"Agent built with {len(all_tools)} tools:", [t.name for t in all_tools])


Agent built with 5 tools: ['check_showtimes', 'log_time', 'get_timesheet', 'get_project_summary', 'list_projects']


---
## 5. Ask it something -- a tiny input loop

Type a question, see which tool(s) the model reaches for, and watch the agent answer using
whichever live servers actually connected. Type `quit` to stop.


In [ ]:
while True:
    question = input("\nAsk the fleet agent (or 'quit'): ").strip()
    if question.lower() in {"quit", "exit", ""}:
        break
    try: 
        result = await agent.ainvoke({"messages": [("user", question)]})
        for msg in result["messages"]:
            if getattr(msg, "tool_calls", None):
                for tc in msg.tool_calls:
                    print(f"  [tool call] {tc['name']}({tc['args']})")
        print("Agent:", result["messages"][-1].content)
    except Exception as e:
        print(f"Call skipped -- {type(e).__name__}: {e}")


Agent: I’m not sure which “Time MCP” server you mean — can you clarify? A few common possibilities and what you can do on each:

1) TimeMC (a Minecraft PVP/HCF-style network)
- Typical actions: join survival/PvP/factions arenas, claim land, create/join factions, use kits, open crates, participate in events, buy ranks/keys, trade, view leaderboards.
- Common commands: /warp, /home, /sethome, /tpa, /msg, /f create, /f invite, /claim, /kit, /vote.
- Admining: manage bans/whitelist, set up crates/menus, run events, use permissions/rank plugins.

2) MCP (Mod Coder Pack / Minecraft modding development server)
- Typical actions: decompile/deobfuscate Minecraft code, build and test mods, run a local server to test modded gameplay, debug, create Forge/Fabric mods, patch mappings, and share a dev server with other modders.
- Admining: configure mod list, sync server/client mods, tweak configs, profile performance.

3) Generic “MCP” as a hosted Minecraft control panel / private server
- Typical a

## Summary

- One agent, three independent MCP connections: local stdio, a free public HTTP server, your own
  deployed server — merged into a single tool list with per-server `try/except` so one outage
  doesn't sink the demo.
- The local connection is the one verified by a real run during authoring; the two remote ones
  are correct-per-the-docs but could not be executed from the authoring sandbox's restricted
  network — expected to run clean from a normal machine, including yours.
- This is the "independent connections" shape from the main notebook's `ClientGroup` section,
  done by hand with per-server error handling instead, because that's what survives a live demo
  in front of an audience when a venue's wifi drops one server mid-class.
